# Bus Corridor Simulator

A six-stop urban bus route with ten scheduled buses in the reference run. This notebook measures how a temporary road delay spreads through the corridor and affects passenger waiting. The first experiment introduces a road delay. The second compares immediate release with capped holding before Market using the same disruption and passenger inputs.

## 1. Problem and question

When one bus loses time on the road, passengers accumulate at the next stop. The later bus can therefore board more people and occupy the berth for longer. A following bus that reaches the occupied berth waits in a separate bus queue.

The question is whether holding buses briefly before a congested stop can reduce passenger delay. This checkpoint tests how one capped policy behaves in a fixed disruption before comparing average effects across random scenarios.

**Route:** Origin → Market → School → Junction → Health Centre → Terminal. Stop names describe a synthetic corridor. Distances and demand are assumed, with no claim that they describe a measured route.

## 2. Model, assumptions and equations

Each stop has one berth and a first-in-first-out bus queue. Passengers enter at an assigned origin and leave at an assigned downstream destination. At the start of service, alighting passengers leave and waiting passengers board. The bus occupies the berth for the remaining dwell. Passengers arriving later wait for another bus. The model uses unlimited seating and one direction of travel.

For bus $i$ on link $j$, its nominal arrival is departure time plus running time. A bus cannot pass the bus ahead on a link, so the realised arrival also respects the preceding realised link arrival $	au_{i-1,j}$:

$$\tau_{i,j}=\max\bigl(t_{i,j}^{\mathrm{departure}}+r_{i,j},\ \tau_{i-1,j}+1\bigr).$$

The second term is omitted for the first bus on the link. The one-second separation is an assumed minimum, and time added by this rule counts as running time. At a stop, boarding/alighting counts $B$ and $A$ determine the duration in seconds:

$$d=d_0+\beta_b B+\beta_a A.$$

The example uses the rural Kandy reference coefficients $d_0=5.264$ s, $\beta_b=2.136$ s per boarding and $\beta_a=1.503$ s per alighting. Their use here is a scenario assumption. Hand checks use integer coefficients. Alighting and boarding timestamps are both defined at service start, so the passenger's measure ends when alighting starts.

$$W=t_{\mathrm{board}}-t_{\mathrm{arrival}},\quad I=t_{\mathrm{alight}}-t_{\mathrm{board}},\quad J=W+I.$$

The control point is a separate FIFO queue at the end of the approach to Market. It precedes the boarding berth and serves no passengers. A bus at the front is ready for a decision at time $t$. If the previous bus actually left at $R$, the observed gap is $g=t-R$:

$$h=\min(h_{\max},\max(0,H-g)),\qquad R_{\mathrm{new}}=t+h.$$

$H$ is the target gap and $h_{\max}$ is the cap. The first bus leaves immediately. A bus can spend additional time waiting behind a held bus. The model records that queue time separately from $h$. The control point and the berth share a route position in the plot. Travel from the control point to the entrance to the berth queue is treated as instantaneous.

The fixed inputs let us audit the controller timing exactly. Repeated stochastic comparisons will build on the same bus and passenger records.

## 3. Implementation

The configuration is a list of stop names, one running time per link, ordered bus dispatches, and a passenger manifest. Number stops from zero. An event heap sorts by time, event priority and insertion order. At a shared timestamp, berth release and arrivals occur before service starts. Each bus visit records its arrival, start of service, boardings, alightings and departure.

Control arrivals and releases use their own events. A released bus then enters the stop queue at the same timestamp.

In [ ]:
from collections import deque
from heapq import heappop, heappush
from itertools import count
from math import isfinite
from statistics import fmean

EVENT_PRIORITY = {
    "berth_release": 0,
    "passenger_arrival": 1,
    "control_arrival": 2,
    "control_release": 3,
    "bus_arrival": 4,
    "service_start": 5,
}
KANDY_DWELL = (5.264, 2.136, 1.503)


def simulate_corridor(
    stops, running_times, dispatches, trips, *, dwell=KANDY_DWELL,
    disruption=None, control_stop=None, target_gap=80, hold_cap=0,
    running_time_draws=None, one_stop_service_times=None
):
    """Run the corridor until its event calendar empties. All times are in seconds.

    The optional one-stop service times are for bus-only queue verification.
    A run with an unfinished passenger raises an error.
    """
    # Each trip contains its arrival time, origin index and destination index.
    if len(stops) != len(running_times) + 1:
        raise ValueError("Give one running time per link between stops.")
    if any(t <= 0 for t in running_times) or tuple(sorted(dispatches)) != tuple(dispatches):
        raise ValueError("Link times must be positive and dispatches must be ordered.")
    if not dispatches or dispatches[0] < 0:
        raise ValueError("Give at least one nonnegative dispatch time.")
    if running_time_draws is not None:
        if len(running_time_draws) != len(dispatches):
            raise ValueError("Give one row of running-time draws per dispatched bus.")
        for bus_times in running_time_draws:
            if len(bus_times) != len(running_times):
                raise ValueError("Give each bus one running-time draw per link.")
            if any(not isfinite(seconds) or seconds <= 0 for seconds in bus_times):
                raise ValueError("Running-time draws must be finite and positive.")
    if one_stop_service_times is not None:
        if len(stops) != 1 or trips:
            raise ValueError("Override service times only in a bus-only, one-stop run.")
        if len(one_stop_service_times) != len(dispatches) or any(
            not isfinite(seconds) or seconds <= 0 for seconds in one_stop_service_times
        ):
            raise ValueError("Give each bus a finite positive service time.")

    fixed_dwell, seconds_per_boarding, seconds_per_alighting = dwell
    if fixed_dwell <= 0 or min(seconds_per_boarding, seconds_per_alighting) < 0:
        raise ValueError("Fixed dwell must be positive and passenger coefficients nonnegative.")
    if disruption is not None:
        delayed_bus, delayed_link, extra_time = disruption
        if not (1 <= delayed_bus <= len(dispatches)
                and 0 <= delayed_link < len(running_times) and extra_time >= 0):
            raise ValueError("Disruption must identify an existing bus, link and nonnegative delay.")

    if control_stop is not None and not 0 < control_stop < len(stops) - 1:
        raise ValueError("Put the control point before a downstream boarding stop.")
    if target_gap <= 0 or hold_cap < 0:
        raise ValueError("Target gap must be positive and the hold cap nonnegative.")
    if control_stop is None and hold_cap > 0:
        raise ValueError("Choose a control stop to enable holding.")

    calendar = []
    sequence = count()
    trace = []
    # Buses queue for a berth. Passengers wait separately on the platform.
    bus_queues = [deque() for _ in stops]
    platform_queues = [deque() for _ in stops]
    berth_bus = [None for _ in stops]
    last_link_arrival = [float("-inf") for _ in running_times]
    bus_visits = {}
    incoming_road_times = {}
    passengers = []
    onboard = {bus: [] for bus in range(1, len(dispatches) + 1)}
    control_queue = deque()
    control_times = {}
    active_control_bus = None
    last_control_release = None

    def schedule(time, event, subject, stop):
        heappush(calendar, (time, EVENT_PRIORITY[event], next(sequence), event, subject, stop))

    def release_next_bus(now):
        nonlocal active_control_bus
        if active_control_bus is not None or not control_queue:
            return
        bus = control_queue.popleft()
        active_control_bus = bus
        times = control_times[bus]
        times["control_queue_wait"] = now - times["control_arrival"]

        hold = 0
        if last_control_release is not None:
            observed_gap = now - last_control_release
            hold = min(hold_cap, max(0, target_gap - observed_gap))
        times["hold"] = hold
        schedule(now + hold, "control_release", bus, control_stop)

    for bus, dispatch in enumerate(dispatches, start=1):
        schedule(dispatch, "bus_arrival", bus, 0)
    for rider, (arrival, origin, destination) in enumerate(trips):
        if arrival < 0 or not 0 <= origin < destination < len(stops):
            raise ValueError("Passenger trips need a nonnegative time and a downstream destination.")
        passengers.append({
            "arrival": arrival, "origin": origin, "destination": destination,
            "board": None, "alight": None, "bus": None,
        })
        schedule(arrival, "passenger_arrival", rider, origin)

    previous_time = 0
    while calendar:
        now, _, _, event, subject, stop = heappop(calendar)
        assert now >= previous_time
        previous_time = now
        trace.append((now, event, subject, stop))
        queue = bus_queues[stop]

        if event == "passenger_arrival":
            platform_queues[stop].append(subject)

        elif event == "control_arrival":
            control_times[subject] = {"control_arrival": now}
            control_queue.append(subject)
            release_next_bus(now)

        elif event == "control_release":
            assert active_control_bus == subject
            control_times[subject]["control_release"] = now
            last_control_release = now
            active_control_bus = None
            schedule(now, "bus_arrival", subject, stop)
            release_next_bus(now)

        elif event == "bus_arrival":
            visit = {"arrival": now}
            if stop > 0:
                visit.update(incoming_road_times.pop((subject, stop)))
            if stop == control_stop:
                visit.update(control_times[subject])
            bus_visits[subject, stop] = visit
            queue.append(subject)
            if berth_bus[stop] is None and len(queue) == 1:
                schedule(now, "service_start", subject, stop)

        elif event == "service_start":
            front_bus = queue.popleft()
            assert front_bus == subject and berth_bus[stop] is None
            berth_bus[stop] = subject

            remaining = []
            alightings = 0
            for rider in onboard[subject]:
                passenger = passengers[rider]
                if passenger["destination"] == stop:
                    passenger["alight"] = now
                    alightings += 1
                else:
                    remaining.append(rider)

            boardings = len(platform_queues[stop])
            while platform_queues[stop]:
                rider = platform_queues[stop].popleft()
                passengers[rider]["board"] = now
                passengers[rider]["bus"] = subject
                remaining.append(rider)
            onboard[subject] = remaining

            duration = (
                fixed_dwell
                + seconds_per_boarding * boardings
                + seconds_per_alighting * alightings
            )
            if one_stop_service_times is not None:
                duration = one_stop_service_times[subject - 1]
            visit = bus_visits[subject, stop]
            visit.update(
                service_start=now, boardings=boardings, alightings=alightings,
                berth_queue_wait=now - visit["arrival"], dwell_seconds=duration,
            )
            schedule(now + duration, "berth_release", subject, stop)

        elif event == "berth_release":
            assert berth_bus[stop] == subject
            bus_visits[subject, stop]["departure"] = now
            berth_bus[stop] = None
            if queue:
                schedule(now, "service_start", queue[0], stop)
            if stop < len(running_times):
                travel_seconds = running_times[stop]
                if running_time_draws is not None:
                    # Bus IDs start at one; rows and link indices start at zero.
                    travel_seconds = running_time_draws[subject - 1][stop]
                disruption_seconds = 0
                if disruption is not None and subject == delayed_bus and stop == delayed_link:
                    disruption_seconds = extra_time
                free_arrival = now + travel_seconds + disruption_seconds
                # Keep buses in departure order on each link.
                next_arrival = max(free_arrival, last_link_arrival[stop] + 1)
                last_link_arrival[stop] = next_arrival
                next_stop = stop + 1
                incoming_road_times[subject, next_stop] = {
                    "sampled_road_seconds": travel_seconds,
                    "road_disruption_seconds": disruption_seconds,
                    "no_overtaking_seconds": next_arrival - free_arrival,
                }
                arrival_event = "control_arrival" if next_stop == control_stop else "bus_arrival"
                schedule(next_arrival, arrival_event, subject, next_stop)

    completed = sum(rider["alight"] is not None for rider in passengers)
    still_waiting = sum(map(len, platform_queues))
    assert completed + still_waiting + sum(map(len, onboard.values())) == len(passengers)
    assert not any(onboard.values()), "All boarded passengers must alight by the terminal."
    assert len(bus_visits) == len(stops) * len(dispatches)
    assert not control_queue and active_control_bus is None
    assert not incoming_road_times
    if still_waiting:
        raise ValueError(
            f"Only {completed} of {len(passengers)} passengers completed; "
            f"{still_waiting} still waiting when service ended. Extend the dispatch schedule."
        )
    return bus_visits, passengers, trace

## 4. Verification

The one-berth case uses two buses to isolate the expected 20-second wait, then a third bus to exercise the exact timestamp tie. Bus 1 arrives at 70 s and departs at 100 s. Bus 2 arrives at 80 s and keeps its place when Bus 3 arrives at the moment the berth frees. The three-stop test checks a complete journey and two consecutive berth queues in the same engine.

In [ ]:
# One berth, fixed 30 s dwell. Bus 3 arrives at exactly the first berth release.
visits, _, trace = simulate_corridor(
    ("Test stop",), (), (70, 80, 100), (), dwell=(30, 0, 0)
)
assert visits[1, 0]["departure"] == 100
assert visits[2, 0]["service_start"] - visits[2, 0]["arrival"] == 20
assert visits[3, 0]["service_start"] == 130
assert [event for time, event, _, _ in trace if time == 100] == [
    "berth_release", "bus_arrival", "service_start",
]
print("PASS: 20 s berth wait and FIFO at a simultaneous release and arrival.")

# Hand trace with d = 10 + 2B + A, two road links of 20 and 30 seconds.
# Passenger 1 boards at 0 s; Passenger 2 arrives at 30 s and boards at 32 s.
# Both reach their destination at 74 s. The second bus queues at both downstream stops.
visits, riders, _ = simulate_corridor(
    ("Origin", "Middle", "Terminal"), (20, 30), (0, 5),
    ((0, 0, 2), (30, 1, 2)), dwell=(10, 2, 1)
)
assert visits[1, 1]["arrival"] == 32
assert visits[2, 1]["service_start"] - visits[2, 1]["arrival"] == 2
assert visits[2, 2]["service_start"] - visits[2, 2]["arrival"] == 2
assert [(rider["board"], rider["alight"]) for rider in riders] == [(0, 74), (32, 74)]
assert all(rider["bus"] == 1 for rider in riders)
print("PASS: full trips take 74 s and 44 s, with 0 s and 2 s platform waits.")

In [ ]:
# Approach arrivals are 11, 51, 56 and 151 s. The target gap is 60 s and
# the cap is 15 s. Bus 3 reaches the front only after Bus 2 releases at 66 s.
control_visits, _, _ = simulate_corridor(
    ("Origin", "Market", "Terminal"), (10, 10), (0, 40, 45, 140), (),
    dwell=(1, 0, 0), control_stop=1, target_gap=60, hold_cap=15,
)
expected_control = (
    (11, 11, 0, 0), (51, 66, 0, 15), (56, 81, 10, 15), (151, 151, 0, 0),
)
timing_fields = ("control_arrival", "control_release", "control_queue_wait", "hold")
for bus, expected in enumerate(expected_control, start=1):
    visit = control_visits[bus, 1]
    assert tuple(visit[field] for field in timing_fields) == expected
print("PASS: first release, cap, FIFO control queue and long-gap release.")

In [ ]:
# A passenger arriving after the last boarding opportunity must cause an error.
try:
    simulate_corridor(("Origin", "Terminal"), (10,), (0,), ((1, 0, 1),), dwell=(1, 0, 0))
except ValueError as error:
    assert "1 still waiting" in str(error)
else:
    raise AssertionError("An unfinished passenger was silently accepted.")
print("PASS: a short dispatch schedule is rejected with the unfinished passenger count.")

## 5. Reference corridor experiment

Run ten buses at 80-second dispatch intervals. Generate a fixed passenger schedule at the five boarding stops and assign downstream destinations. Replay it twice. In the disrupted run, Bus 4 gets an additional 100 seconds on the first road link. All other specified inputs match.

These runs illustrate what a disturbance does to service. They do not estimate an average causal effect over random conditions.

In [ ]:
STOPS = ("Origin", "Market", "School", "Junction", "Health Centre", "Terminal")
DISTANCE_KM = (0.0, 0.5, 1.1, 1.7, 2.3, 2.9)
RUNNING_TIMES = (60, 70, 75, 70, 80)
DISPATCHES = tuple(range(0, 800, 80))
ARRIVAL_INTERVALS = (13, 8, 10, 11, 12)

# Stagger the first arrivals by stop. Two in every three riders travel to the
# terminal, and the third rider travels one stop. This is a scenario assumption.
TRIPS = []
for origin, interval in enumerate(ARRIVAL_INTERVALS):
    for number, arrival in enumerate(range(5 + 7 * origin, 635, interval)):
        destination = origin + 1 if number % 3 == 0 else len(STOPS) - 1
        TRIPS.append((arrival, origin, destination))

runs = {
    "Normal service": simulate_corridor(STOPS, RUNNING_TIMES, DISPATCHES, TRIPS),
    "Road delay on Bus 4": simulate_corridor(
        STOPS, RUNNING_TIMES, DISPATCHES, TRIPS, disruption=(4, 0, 100)
    ),
}

print(f"{len(STOPS)} stops | {len(DISPATCHES)} buses | {len(TRIPS)} passenger arrivals per run")
print(f"{'Scenario':<24} {'Completed':>9} {'Wait (s)':>10} {'Journey (s)':>12} {'Berth wait (s)':>15}")
for label, (visits, riders, _) in runs.items():
    assert all(rider["alight"] is not None for rider in riders), "Extend service to finish this cohort."
    mean_wait = fmean(rider["board"] - rider["arrival"] for rider in riders)
    mean_journey = fmean(rider["alight"] - rider["arrival"] for rider in riders)
    mean_berth_wait = fmean(visit["service_start"] - visit["arrival"] for visit in visits.values())
    print(f"{label:<24} {len(riders):9d} {mean_wait:10.1f} {mean_journey:12.1f} {mean_berth_wait:15.1f}")

print("\nMarket stop, road-delay run")
for bus in (4, 5):
    visit = runs["Road delay on Bus 4"][0][bus, 1]
    wait = visit["service_start"] - visit["arrival"]
    print(f"Bus {bus}: {visit['boardings']} boardings, {wait:.1f} s waiting for its berth")

### Time–distance figure

Each line is one bus. Flat segments show time at a stop, including berth wait and dwell. The highlighted buses are the delayed Bus 4 and its follower. Panels use the same stop positions and time scale, so their gaps and travel patterns are directly comparable.

Running the figure cells saves two separate PNGs: [corridor-time-distance.png](../figures/corridor-time-distance.png) and [holding-comparison.png](../figures/holding-comparison.png).

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

figure, axes = plt.subplots(1, 2, figsize=(12.5, 5.2), sharex=True, sharey=True)
palette = {4: "#b54826", 5: "#176789"}

for axis, (label, (visits, _, _)) in zip(axes, runs.items()):
    for bus in range(1, len(DISPATCHES) + 1):
        times, positions = [], []
        for index, distance in enumerate(DISTANCE_KM):
            visit = visits[bus, index]
            times.extend((visit["arrival"] / 60, visit["departure"] / 60))
            positions.extend((distance, distance))
        axis.plot(
            times, positions,
            color=palette.get(bus, "#a9b4b8"),
            linewidth=2.6 if bus in palette else 1.35,
            zorder=3 if bus in palette else 2,
        )
    axis.set_title(label, loc="left", fontsize=13, fontweight="bold")
    axis.set_xlabel("Time since first dispatch (min)")
    axis.set_yticks(DISTANCE_KM, STOPS)
    axis.grid(axis="y", color="#dce4e5", linewidth=0.8)
    axis.spines[["top", "right"]].set_visible(False)
    axis.set_ylim(-0.16, 3.06)

# Label the berth wait from the recorded event times.
queued = runs["Road delay on Bus 4"][0][5, 1]
queue_midpoint = (queued["arrival"] + queued["service_start"]) / 120
queue_seconds = queued["service_start"] - queued["arrival"]
axes[1].annotate(
    f"Bus 5 queues for {queue_seconds:.0f} s",
    xy=(queue_midpoint, DISTANCE_KM[1]), xytext=(8.8, 0.15),
    fontsize=9, color="#17455c",
    bbox={"boxstyle": "round,pad=0.25", "fc": "white", "ec": "#dce4e5"},
    arrowprops={"arrowstyle": "-", "color": "#176789", "lw": 1.1},
)
axes[0].set_ylabel("Position along the route")
legend = [Line2D([0], [0], color=color, lw=2.6, label=f"Bus {bus}")
          for bus, color in palette.items()]
legend.append(Line2D([0], [0], color="#a9b4b8", lw=1.35, label="Other buses"))
figure.suptitle("How a road delay changes bus spacing", x=0.12, y=0.96, ha="left",
                fontsize=19, fontweight="bold")
figure.legend(handles=legend, loc="lower center", ncol=3, frameon=False)
figure.tight_layout(rect=(0, 0.07, 1, 0.92))
Path("figures").mkdir(exist_ok=True)
figure.savefig("figures/corridor-time-distance.png", dpi=180, bbox_inches="tight")
plt.close(figure)
print("Saved figures/corridor-time-distance.png")

### Capped holding under the same disruption

Keep the 100-second road delay, ten buses and all 296 passenger trips in both runs. Compare the no-holding case with an 80-second target gap and a 20-second cap before Market. A separate zero-cap check compares every stop and passenger record against the original runs.

In [ ]:
CONTROL_STOP = 1  # Immediately before Market, after the first road link.
TARGET_GAP = 80
HOLD_CAP = 20
HOLDING_LABEL = f"{HOLD_CAP:g} s holding cap"
ROAD_DELAY = (4, 0, 100)
control_fields = {"control_arrival", "control_release", "control_queue_wait", "hold"}

# With the cap at zero, the control point must preserve the two existing runs.
for label, disruption in (("Normal service", None), ("Road delay on Bus 4", ROAD_DELAY)):
    original_visits, original_riders, _ = runs[label]
    zero_visits, zero_riders, _ = simulate_corridor(
        STOPS, RUNNING_TIMES, DISPATCHES, TRIPS, disruption=disruption,
        control_stop=CONTROL_STOP, target_gap=TARGET_GAP, hold_cap=0,
    )
    zero_stop_times = {
        key: {field: value for field, value in visit.items() if field not in control_fields}
        for key, visit in zero_visits.items()
    }
    assert zero_stop_times == original_visits and zero_riders == original_riders
    assert all(visit.get("hold", 0) == 0 and visit.get("control_queue_wait", 0) == 0
               for visit in zero_visits.values())
print("PASS: a zero cap reproduces all stop times and passenger journeys in both baseline runs.")

policy_runs = {
    "No holding": runs["Road delay on Bus 4"],
    HOLDING_LABEL: simulate_corridor(
        STOPS, RUNNING_TIMES, DISPATCHES, TRIPS, disruption=ROAD_DELAY,
        control_stop=CONTROL_STOP, target_gap=TARGET_GAP, hold_cap=HOLD_CAP,
    ),
}


def policy_metrics(result):
    visits, riders, _ = result
    assert all(rider["alight"] is not None for rider in riders), "Compare the full passenger cohort."
    market_visits = [visits[bus, CONTROL_STOP] for bus in range(1, len(DISPATCHES) + 1)]
    return {
        "platform_wait": fmean(rider["board"] - rider["arrival"] for rider in riders),
        "journey_time": fmean(rider["alight"] - rider["arrival"] for rider in riders),
        "market_berth_wait": fmean(v["service_start"] - v["arrival"] for v in market_visits),
        "control_delay": fmean(v.get("hold", 0) + v.get("control_queue_wait", 0)
                              for v in market_visits),
    }


policy_results = {label: policy_metrics(result) for label, result in policy_runs.items()}
print(f"\nSame road delay, same {len(TRIPS)} passengers")
print(f"{'Policy':<20} {'Platform (s)':>13} {'Journey (s)':>12} {'Market queue (s)':>17} {'Control (s)':>12}")
for label, metrics in policy_results.items():
    print(f"{label:<20} {metrics['platform_wait']:13.2f} {metrics['journey_time']:12.2f}"
          f" {metrics['market_berth_wait']:17.2f} {metrics['control_delay']:12.2f}")
print("Platform and journey are per passenger. Market queue and control are per bus.")

In [ ]:
policy_figure, policy_axes = plt.subplots(1, 2, figsize=(12.5, 5.2), sharex=True, sharey=True)
for axis, (label, (visits, _, _)) in zip(policy_axes, policy_runs.items()):
    for bus in range(1, len(DISPATCHES) + 1):
        times, positions = [], []
        for stop, distance in enumerate(DISTANCE_KM):
            visit = visits[bus, stop]
            if "control_arrival" in visit:
                # The control point and the Market berth are shown at the same position.
                times.append(visit["control_arrival"] / 60)
                positions.append(distance)
            times.extend((visit["arrival"] / 60, visit["departure"] / 60))
            positions.extend((distance, distance))
        axis.plot(times, positions, color=palette.get(bus, "#a9b4b8"),
                  linewidth=2.6 if bus in palette else 1.35, zorder=3 if bus in palette else 2)
    axis.set_title(label, loc="left", fontsize=13, fontweight="bold")
    axis.set_xlabel("Time since first dispatch (min)")
    axis.set_yticks(DISTANCE_KM, STOPS)
    axis.grid(axis="y", color="#dce4e5", linewidth=0.8)
    axis.spines[["top", "right"]].set_visible(False)
    axis.set_ylim(-0.16, 3.06)

uncontrolled = policy_runs["No holding"][0][5, CONTROL_STOP]
queue_seconds = uncontrolled["service_start"] - uncontrolled["arrival"]
queue_midpoint = (uncontrolled["arrival"] + uncontrolled["service_start"]) / 120
policy_axes[0].annotate(
    f"Bus 5 queues for {queue_seconds:.0f} s",
    xy=(queue_midpoint, DISTANCE_KM[CONTROL_STOP]), xytext=(8.8, 0.15),
    fontsize=9, color="#17455c",
    bbox={"boxstyle": "round,pad=0.25", "fc": "white", "ec": "#dce4e5"},
    arrowprops={"arrowstyle": "-", "color": "#176789", "lw": 1.1},
)
market = policy_runs[HOLDING_LABEL][0][5, CONTROL_STOP]
hold_start = market["control_release"] - market["hold"]
hold_end = market["control_release"]
hold_midpoint = (hold_start + hold_end) / 120
policy_axes[1].annotate(
    f"Bus 5 held for {market['hold']:.0f} s",
    xy=(hold_midpoint, DISTANCE_KM[CONTROL_STOP]), xytext=(8.8, 0.15),
    fontsize=9, color="#17455c",
    bbox={"boxstyle": "round,pad=0.25", "fc": "white", "ec": "#dce4e5"},
    arrowprops={"arrowstyle": "-", "color": "#176789", "lw": 1.1},
)
policy_axes[0].set_ylabel("Position along the route")
policy_figure.suptitle("Departure policies under the same road disruption",
                      x=0.12, y=0.96, ha="left", fontsize=17, fontweight="bold")
policy_figure.legend(handles=legend, loc="lower center", ncol=3, frameon=False)
policy_figure.tight_layout(rect=(0, 0.07, 1, 0.92))
policy_figure.savefig("figures/holding-comparison.png", dpi=180, bbox_inches="tight")
plt.close(policy_figure)
print("Saved figures/holding-comparison.png")

### Seeded passenger demand

Model each boarding stop as a constant-rate Poisson arrival stream. If the rate is $\lambda$ passengers per minute and $U$ is uniform on $(0,1)$, the time in seconds until the next arrival is

$$\Delta t = -\frac{60}{\lambda}\ln(1-U).$$

The expected inter-arrival time is $60/\lambda$ seconds. Keep arrivals before the cutoff and skip stops with zero demand. A fixed seed gives a repeatable passenger manifest. Future policy comparisons can pass the same manifest to both runs.

The sample uses the same mean arrival rates as the fixed-arrival case. It keeps the existing destination assignment rule, so the new randomness comes from passenger arrival times.

In [ ]:
from math import isfinite, log1p
from random import Random


def poisson_arrivals(rate_per_minute, end_seconds, rng):
    """Return one stop's arrival times in seconds before the cutoff."""
    if not isfinite(rate_per_minute) or rate_per_minute < 0:
        raise ValueError("The arrival rate must be finite and nonnegative.")
    if not isfinite(end_seconds) or end_seconds < 0:
        raise ValueError("The arrival cutoff must be finite and nonnegative.")
    if rate_per_minute == 0:
        return []

    mean_gap = 60 / rate_per_minute
    arrival_time = 0.0
    arrivals = []
    while arrival_time < end_seconds:
        draw = rng.random()
        if draw == 0:
            continue  # Use the open interval so every gap is positive.
        arrival_time += -mean_gap * log1p(-draw)
        if arrival_time < end_seconds:
            arrivals.append(arrival_time)
    return arrivals


def generate_passenger_trips(rates_per_minute, end_seconds, seed):
    """Return (arrival, origin, destination) for the corridor."""
    rng = Random(seed)
    terminal = len(rates_per_minute)
    trips = []
    for origin, rate in enumerate(rates_per_minute):
        for rider, arrival in enumerate(poisson_arrivals(rate, end_seconds, rng)):
            # Match the fixed-input case: every third rider makes a one-stop trip.
            destination = origin + 1 if rider % 3 == 0 else terminal
            trips.append((arrival, origin, destination))
    return sorted(trips, key=lambda trip: trip[0])

In [ ]:
DEMAND_RATES = tuple(60 / interval for interval in ARRIVAL_INTERVALS)
DEMAND_CUTOFF = 635
DEMAND_SEED = 42
sample_trips = generate_passenger_trips(DEMAND_RATES, DEMAND_CUTOFF, DEMAND_SEED)
assert sample_trips == generate_passenger_trips(DEMAND_RATES, DEMAND_CUTOFF, DEMAND_SEED)
assert sample_trips != generate_passenger_trips(DEMAND_RATES, DEMAND_CUTOFF, DEMAND_SEED + 1)
assert all(0 < time < DEMAND_CUTOFF and 0 <= origin < destination < len(STOPS)
           for time, origin, destination in sample_trips)
assert poisson_arrivals(0, DEMAND_CUTOFF, Random(DEMAND_SEED)) == []
print(f"PASS: seed {DEMAND_SEED} reproduces all {len(sample_trips)} passenger trips.")

# At 6 arrivals per minute, a long sample should average close to a 10 s gap.
reference_rate = 6
arrivals = poisson_arrivals(reference_rate, 50_000, Random(DEMAND_SEED))
gaps = [second - first for first, second in zip([0.0] + arrivals, arrivals)]
expected_gap = 60 / reference_rate
observed_gap = fmean(gaps)
assert len(gaps) > 4_000 and all(gap > 0 for gap in gaps)
assert abs(observed_gap - expected_gap) < 0.05 * expected_gap
print(f"PASS: mean gap {observed_gap:.3f} s vs {expected_gap:.1f} s from {len(gaps)} gaps (5% tolerance).")

_, sample_riders, _ = simulate_corridor(STOPS, RUNNING_TIMES, DISPATCHES, sample_trips)
assert all(rider["alight"] is not None for rider in sample_riders)
print("PASS: the six-stop model completed every generated trip.")

### Seeded link running times

For each bus and link, draw a positive free-running time from a lognormal distribution. Given the desired arithmetic mean $m$ in seconds and coefficient of variation $c$ (standard deviation divided by mean), the parameters on the log scale are

$$\sigma_{\log}^2=\ln(1+c^2),\qquad \mu_{\log}=\ln(m)-\tfrac12\sigma_{\log}^2.$$

These parameters keep the expected free-running time at $m$ as variability changes. Use the existing link times as the means. A coefficient of zero gives exactly those fixed times. The matrix is indexed by bus and link, so future policy runs can share identical draws even when their event order differs. The no-passing rule can still increase a bus's realised running time after the draw.

In [ ]:
from math import log, sqrt
from statistics import pstdev


def sample_link_running_times(mean_seconds, coefficient_of_variation, bus_count, seed):
    """Return a tuple of free-running times by bus and link."""
    if not mean_seconds or any(not isfinite(mean) or mean <= 0 for mean in mean_seconds):
        raise ValueError("Give each link a finite positive mean running time.")
    if not isfinite(coefficient_of_variation) or coefficient_of_variation < 0:
        raise ValueError("The coefficient of variation must be finite and nonnegative.")
    if bus_count < 1:
        raise ValueError("Sample running times for at least one bus.")

    if coefficient_of_variation == 0:
        return (tuple(mean_seconds),) * bus_count

    rng = Random(seed)
    log_variance = log1p(coefficient_of_variation ** 2)
    log_sd = sqrt(log_variance)
    log_means = [log(mean) - log_variance / 2 for mean in mean_seconds]
    return tuple(
        tuple(rng.lognormvariate(log_mean, log_sd) for log_mean in log_means)
        for _ in range(bus_count)
    )

In [ ]:
from math import isclose

TRAVEL_SEED = 73
for target_cv in (0.15, 0.45):
    sample = [row[0] for row in sample_link_running_times((60,), target_cv, 10_000, TRAVEL_SEED)]
    observed_mean = fmean(sample)
    observed_cv = pstdev(sample) / observed_mean
    assert all(seconds > 0 for seconds in sample)
    assert abs(observed_mean - 60) < 1.8  # 3% of the target mean.
    assert abs(observed_cv - target_cv) < 0.02
    print(f"PASS: target CV {target_cv:.2f}, sample mean {observed_mean:.2f} s, sample CV {observed_cv:.3f}.")

fixed_draws = sample_link_running_times(RUNNING_TIMES, 0, len(DISPATCHES), TRAVEL_SEED)
assert fixed_draws == (RUNNING_TIMES,) * len(DISPATCHES)
assert simulate_corridor(STOPS, RUNNING_TIMES, DISPATCHES, TRIPS,
                         running_time_draws=fixed_draws) == runs["Normal service"]
print("PASS: zero variability reproduces the fixed-time corridor.")

sampled_draws = sample_link_running_times(RUNNING_TIMES, 0.15, len(DISPATCHES), TRAVEL_SEED)
assert sampled_draws == sample_link_running_times(RUNNING_TIMES, 0.15, len(DISPATCHES), TRAVEL_SEED)
visits, riders, _ = simulate_corridor(
    STOPS, RUNNING_TIMES, DISPATCHES, TRIPS, running_time_draws=sampled_draws
)
first_link_time = visits[1, 1]["arrival"] - visits[1, 0]["departure"]
assert isclose(first_link_time, sampled_draws[0][0])
assert all(rider["alight"] is not None for rider in riders)
print("PASS: seeded draws reproduce, enter the event schedule, and complete the passenger cohort.")

### Analytical benchmarks through the common engine

For the one-berth M/M/1 mode, use independent exponential inter-arrivals and service with $lambda=0.5$ buses/min and $mu=1$ bus/min. The steady-state mean bus queue wait is $lambda/[mu(mu-lambda)]=1$ minute. Discard the first 1,000 buses of each run to reduce startup bias. Compare eight means, each from the following 4,000 bus arrivals, with the analytical 60 seconds using a tolerance of 4.8 seconds.

In [ ]:
from itertools import accumulate

def verify_mm1(replications=8, warmup_buses=1_000, measured_buses=4_000):
    # Rates in buses per second. The one-stop engine handles all FIFO events.
    arrival_rate, service_rate = 0.5 / 60, 1 / 60
    total_buses = warmup_buses + measured_buses
    expected = arrival_rate / (service_rate * (service_rate - arrival_rate))
    estimates = []
    for rep in range(replications):
        arrival_rng = Random(1100 + rep)
        service_rng = Random(2100 + rep)
        arrivals = tuple(accumulate(arrival_rng.expovariate(arrival_rate)
                                    for _ in range(total_buses)))
        service = tuple(service_rng.expovariate(service_rate) for _ in range(total_buses))
        visits, _, _ = simulate_corridor(
            ("One berth",), (), arrivals, (), one_stop_service_times=service
        )
        estimates.append(fmean(visits[bus, 0]["berth_queue_wait"]
                               for bus in range(warmup_buses + 1, total_buses + 1)))
    return expected, fmean(estimates), replications * measured_buses



def record_benchmark(label, result, tolerance):
    expected, actual, observations = result
    assert abs(actual - expected) <= tolerance, (label, expected, actual, tolerance)
    print(f"PASS: {label}: expected {expected:.2f} s, simulated {actual:.2f} s "
          f"(tolerance ±{tolerance:.1f} s, {observations:,} observations).")


record_benchmark("M/M/1 berth queue", verify_mm1(), 4.8)

## 6. Discussion and limitations

The road-delay runs share a dispatch schedule and a complete passenger cohort. The controlled case shows how a lower berth queue can coexist with an upstream delay. Read the passenger measures alongside both bus measures.

With the same road delay, Bus 5 spends 57.9 s waiting for the Market berth with immediate release and 37.9 s with the cap. The controller holds that bus for 20.0 s. Across the same 296 passengers, mean platform wait is 83.74 s without holding and 84.28 s with the cap. Mean journey times are 382.18 s and 382.63 s, respectively. These are results for one fixed scenario.

The route, demand and running times are synthetic. Each scenario is a single deterministic run. The next experiment varies passenger arrivals and running times, compares matched runs and reports confidence intervals for the policy differences.

## 7. References

- [Herath et al. (2024), dwell equation 9 and table 3](https://www.jstage.jst.go.jp/article/easts/15/0/15_2109/_pdf/-char/en)
- [Jayasinghe et al. (2023), Galle Road bus operations](https://doi.org/10.4038/jsalt.v3i1.57)
- [Shen et al. (2026), headway irregularity and stop queues](https://doi.org/10.1016/j.tra.2025.104752)